# Nemotron 3 Nano — SFT → GRPO RLVR Pipeline (RTX 6000 Pro)

Single-GPU adaptation of `sft/nvidia-nemotron-v7-5.ipynb` for one **RTX 6000 Pro
(Blackwell, ~96 GB VRAM)**. Implements the SFT → GRPO RLVR pipeline from
[jhyland01/kaggle_nemotron](https://github.com/jhyland01/kaggle_nemotron) and
the Dr. GRPO / DAPO upgrades in this repo's `post-training/nemo-v11/v12`
notebooks.

## Pipeline

1. **SFT warm-start** on `data/merged_cot_final.csv` — teach
   `<think>...</think>\boxed{}` format + base reasoning.
2. **GRPO RLVR** on `data/src/train.csv` (raw competition prompts + answers) —
   binary verifier reward, no reference model (`beta=0`).
3. **Package** `submission.zip` (LoRA adapter only, rank ≤ 32).

## What changed vs `nvidia-nemotron-v7-5.ipynb`

| Area | Kaggle v7-5 | RTX 6000 Pro v13 |
|---|---|---|
| Stack | Unsloth + offline wheels | Bare HF `transformers` + `peft` + `trl` |
| Paths | `/kaggle/input/...`, `kagglehub` | Local repo paths |
| Phase | SFT only | SFT warm-start → GRPO RLVR |
| LoRA targets | suffix list (includes routable experts) | Regex excluding routable experts + router + `lm_head` |
| Batch sizes | tight (T4/P100) | larger; 96 GB headroom |
| Inference style at train time | N/A | Temperature 1.0 (GRPO rollouts) — eval still greedy |

## Hard constraints (do not change)

- LoRA rank ≤ 32 (eval rejects higher)
- Adapter only: `adapter_config.json` + `adapter_model.safetensors` zipped
- Eval is greedy (`temperature=0`) — model must be confidently correct
- Answer extracted from `\boxed{}` — train so model always ends with it


## Mode & Path Configuration


In [ ]:
import os, sys

os.environ["PYTHONIOENCODING"] = "utf-8"
if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8", errors="strict")
if hasattr(sys.stderr, "reconfigure"):
    sys.stderr.reconfigure(encoding="utf-8", errors="strict")

# Pipeline phase toggles (set to 0 to skip a phase)
RUN_SFT     = 1   # SFT warm-start on merged_cot_final.csv
RUN_GRPO    = 1   # GRPO RLVR on raw train.csv
RUN_PACKAGE = 1   # build submission.zip at the end

# Paths — local repo layout. Override if running elsewhere.
REPO_ROOT = os.environ.get("KAGGLE_NEMO_REPO", r"F:/Hackathons/Kaggle-Nemotron")

BASE_MODEL_NAME       = "nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16"
SFT_DATA_PATH         = os.path.join(REPO_ROOT, "data", "merged_cot_final.csv")
GRPO_DATA_PATH        = os.path.join(REPO_ROOT, "data", "src", "train.csv")

OUTPUT_DIR            = os.path.join(REPO_ROOT, "outputs", "rtx6000_v13")
SFT_ADAPTER_DIR       = os.path.join(OUTPUT_DIR, "sft_adapter")
GRPO_ADAPTER_DIR      = os.path.join(OUTPUT_DIR, "grpo_adapter")
SUBMISSION_DIR        = os.path.join(OUTPUT_DIR, "submission_adapter")
TB_LOG_DIR            = os.path.join(OUTPUT_DIR, "tb_logs")

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(TB_LOG_DIR, exist_ok=True)

SEED = 42
PROMPT_SUFFIX = '\nPlease put your final answer inside `\\boxed{}`. For example: `\\boxed{your answer}`'
MAX_SEQ_LEN_SFT       = 4096       # SFT context window (post-padding)
GRPO_MAX_COMPLETION   = 2048       # GRPO completion budget (per-rollout)
GRPO_MAX_PROMPT       = 2048
LORA_RANK             = 32
LORA_ALPHA            = 64
LORA_DROPOUT          = 0.0

print({
    "RUN_SFT": RUN_SFT, "RUN_GRPO": RUN_GRPO, "RUN_PACKAGE": RUN_PACKAGE,
    "OUTPUT_DIR": OUTPUT_DIR,
    "SFT_DATA_PATH": SFT_DATA_PATH,
    "GRPO_DATA_PATH": GRPO_DATA_PATH,
})


## One-time dependency install

Run this **once per environment** (RTX 6000 Pro Blackwell sm_120/sm_100 needs
recent CUDA + PyTorch nightly for Triton support). Set `INSTALL_DEPS=True`
once, restart kernel, set back to `False`.

Skip if you already have a working env with: `torch>=2.4`, `transformers>=4.46`,
`peft>=0.13`, `trl>=0.16`, `mamba-ssm`, `causal-conv1d`, `bitsandbytes>=0.44`.


In [ ]:
INSTALL_DEPS = False   # set to True only for first run, then restart kernel

if INSTALL_DEPS:
    import subprocess, sys
    pkgs = [
        # Blackwell support: PyTorch 2.4+ with CUDA 12.4+ recommended.
        # If RTX 6000 Pro needs nightly: pip install --pre torch --index-url https://download.pytorch.org/whl/nightly/cu124
        "torch>=2.4",
        "transformers>=4.46",
        "peft>=0.13",
        "trl>=0.16",            # required for loss_type='dapo' / scale_rewards / mask_truncated_completions
        "datasets>=3.0",
        "accelerate>=1.0",
        "bitsandbytes>=0.44",
        "tensorboard",
        "mamba-ssm",            # Nemotron Mamba layers
        "causal-conv1d",        # Mamba Conv1D kernel
        "kagglehub",            # optional model download
        "pandas", "numpy",
    ]
    subprocess.run([sys.executable, "-m", "pip", "install", "-U"] + pkgs, check=True)
    print("Install done. RESTART the kernel before continuing.")
else:
    print("Skipping install (INSTALL_DEPS=False).")


## Model + tokenizer loading

Loads `Nemotron-3-Nano-30B-A3B-BF16` in BF16 with `eager` attention. Total
~60 GB just for weights — leaves ~30+ GB for activations, KV cache, LoRA
state, optimizer.

> No Unsloth: Blackwell PTXAS path quirks make Unsloth fragile here. Bare HF
> is slower but more reliable, and there's enough memory headroom that
> Unsloth memory savings aren't required.


In [ ]:
import torch, gc
from transformers import AutoModelForCausalLM, AutoTokenizer

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

if not torch.cuda.is_available():
    raise RuntimeError("Need CUDA-capable GPU (RTX 6000 Pro Blackwell).")

print(f"GPU: {torch.cuda.get_device_name()}  Total mem: {torch.cuda.get_device_properties(0).total_memory/1e9:.0f} GB")

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Loading base model (bf16, eager attention)...")
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_NAME,
    dtype=torch.bfloat16,
    trust_remote_code=True,
    device_map="auto",
    attn_implementation="eager",
)
model.config.use_cache = False  # required for gradient checkpointing
model.gradient_checkpointing_enable()

total_b = sum(p.numel() for p in model.parameters()) / 1e9
print(f"Model loaded — {total_b:.2f}B params total.  Allocated: {torch.cuda.memory_allocated()/1e9:.2f} GB")


## LoRA target discovery (verify module names before configuring)

Nemotron uses `trust_remote_code=True`, so module names depend on the bundled
modeling code. This cell scans the actual model and groups linear modules
by their parent prefix — confirms which targets exist (`self_attn`, `mamba`,
`shared_expert`, `routable_expert`, etc.) before building the LoRA config.


In [ ]:
import re
from collections import Counter

linear_modules = []
for name, mod in model.named_modules():
    cls = mod.__class__.__name__
    if cls in ("Linear", "Linear4bit", "Linear8bitLt"):
        linear_modules.append(name)

# Suffix breakdown (what kinds of *_proj exist)
suffix_counts = Counter(n.rsplit(".", 1)[-1] for n in linear_modules)
print("Linear suffix counts (top 20):")
for s, c in suffix_counts.most_common(20):
    print(f"  {s:30s} {c}")

# Parent prefix patterns
parent_counts = Counter()
for n in linear_modules:
    parts = n.split(".")
    if len(parts) >= 2:
        parent_counts[parts[-2]] += 1
print("\nParent module counts (top 20):")
for p, c in parent_counts.most_common(20):
    print(f"  {p:30s} {c}")

# Sample a few full names
print("\nSample full names:")
for n in linear_modules[:5] + linear_modules[len(linear_modules)//2:len(linear_modules)//2+5] + linear_modules[-5:]:
    print(f"  {n}")


## LoRA config (rank 32, RSLoRA)

**Targets** (per `CLAUDE.md` LoRA priority table + `WINNING_PLAN.md` §LoRA):

- `self_attn.{q,k,v,o}_proj` — 6 attention layers (quantization-sensitive)
- `mamba.{in,out,x,dt}_proj` — Mamba projections (pre-attn ones most sensitive)
- `shared_expert.{gate,up,down}_proj` — always active, see every token

**Excluded** (intentional):

- Routable experts (only 6/128 active per token → sparse gradient → wasted budget)
- Router / gate weights (NVIDIA freezes during RL — modifying destabilizes routing)
- `lm_head` / embeddings (untied; modifying destabilizes output distribution)


In [ ]:
from peft import LoraConfig, get_peft_model, TaskType

# Regex that matches sensitive targets while EXCLUDING routable_experts + router
target_regex = (
    r".*("
    r"self_attn\.(q|k|v|o)_proj"
    r"|mamba\.(in|out|x|dt)_proj"
    r"|shared_expert\.(gate|up|down)_proj"
    r")$"
)

matched = [n for n in linear_modules if re.match(target_regex, n)]
print(f"Regex matched {len(matched)} modules.")
if not matched:
    print("[WARN] regex matched 0 modules — module names differ. Falling back to suffix-only list.")
    print("       This will also hit routable experts (wasteful). Inspect names above and refine regex.")
    target_modules = ["q_proj","k_proj","v_proj","o_proj","in_proj","out_proj","x_proj","dt_proj","gate_proj","up_proj","down_proj"]
else:
    print("Sample matches:", matched[:6], "...", matched[-3:])
    target_modules = target_regex

lora_config = LoraConfig(
    r              = LORA_RANK,
    lora_alpha     = LORA_ALPHA,
    lora_dropout   = LORA_DROPOUT,
    bias           = "none",
    target_modules = target_modules,
    task_type      = TaskType.CAUSAL_LM,
    use_rslora     = True,          # stable scaling at high rank
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


## SFT data prep (`merged_cot_final.csv`)

Reads `data/merged_cot_final.csv` (expected columns: `prompt`, `cot`, `answer`,
optional `label`). Strips any pre-existing `\boxed{}` from CoT, builds the
canonical assistant turn `<think>{cot}</think>\boxed{answer}`.


In [ ]:
if RUN_SFT:
    import pandas as pd, re
    from datasets import Dataset as HFDataset

    df_sft = pd.read_csv(SFT_DATA_PATH)
    print(f"SFT data: {len(df_sft)} rows.  Columns: {list(df_sft.columns)}")
    required_cols = {"prompt", "cot", "answer"}
    missing = required_cols - set(df_sft.columns)
    if missing:
        raise ValueError(f"Missing required columns in {SFT_DATA_PATH}: {missing}")

    df_sft = df_sft.dropna(subset=["prompt", "cot", "answer"]).reset_index(drop=True)
    df_sft = df_sft.sample(frac=1, random_state=SEED).reset_index(drop=True)

    _BOXED_STRIP = re.compile(r"\\boxed\{[^}]*\}")
    def clean_cot(c):
        c = _BOXED_STRIP.sub("", str(c)).rstrip()
        c = c.replace("<think>", "").replace("</think>", "").strip()
        return c

    records, skipped = [], 0
    for _, row in df_sft.iterrows():
        cot = clean_cot(row["cot"])
        if len(cot) < 5:
            skipped += 1
            continue
        user_content = str(row["prompt"]) + PROMPT_SUFFIX
        assistant_content = f"<think>\n{cot}\n</think>\n\\boxed{{{row['answer']}}}"
        records.append({"messages": [
            {"role": "user",      "content": user_content},
            {"role": "assistant", "content": assistant_content},
        ]})

    sft_dataset = HFDataset.from_list(records)
    print(f"SFT records: {len(records)} (skipped {skipped} short / empty CoT)")


## SFT trainer

Short warm-start (1 epoch, packed). Goal is to imprint chat template and
`<think>...</think>\boxed{}` format before GRPO — *not* to drive accuracy.
GRPO will do the heavy lifting.


In [ ]:
if RUN_SFT:
    import gc, time, torch
    from trl import SFTTrainer, SFTConfig

    def sft_formatting(example):
        msgs = example["messages"]
        if msgs and isinstance(msgs[0], dict):
            convs = [msgs]
        else:
            convs = msgs
        out = []
        for c in convs:
            try:
                t = tokenizer.apply_chat_template(c, tokenize=False,
                                                  add_generation_prompt=False,
                                                  enable_thinking=True)
            except TypeError:
                t = tokenizer.apply_chat_template(c, tokenize=False,
                                                  add_generation_prompt=False)
            out.append(t)
        return out

    sft_args = SFTConfig(
        output_dir                   = os.path.join(OUTPUT_DIR, "sft_run"),
        num_train_epochs             = 1,
        per_device_train_batch_size  = 2,
        gradient_accumulation_steps  = 4,        # effective batch = 8
        learning_rate                = 5e-5,
        lr_scheduler_type            = "cosine",
        warmup_ratio                 = 0.05,
        max_length                   = MAX_SEQ_LEN_SFT,
        packing                      = True,
        optim                        = "paged_adamw_8bit",
        adam_beta1                   = 0.9,
        adam_beta2                   = 0.95,
        adam_epsilon                 = 1e-8,
        weight_decay                 = 0.01,
        max_grad_norm                = 1.0,
        bf16                         = True,
        gradient_checkpointing       = True,
        gradient_checkpointing_kwargs= {"use_reentrant": False},
        logging_steps                = 10,
        logging_dir                  = TB_LOG_DIR,
        report_to                    = "tensorboard",
        save_strategy                = "no",
        dataloader_num_workers       = 4,
        remove_unused_columns        = False,
        seed                         = SEED,
        dataset_num_proc             = 4,
    )

    sft_trainer = SFTTrainer(
        model            = model,
        args             = sft_args,
        train_dataset    = sft_dataset,
        processing_class = tokenizer,
        formatting_func  = sft_formatting,
    )

    torch.cuda.empty_cache(); gc.collect()

    print("Starting SFT warm-start...")
    t0 = time.time()
    sft_trainer.train()
    print(f"SFT done in {(time.time()-t0)/60:.1f} min")

    os.makedirs(SFT_ADAPTER_DIR, exist_ok=True)
    model.save_pretrained(SFT_ADAPTER_DIR)
    tokenizer.save_pretrained(SFT_ADAPTER_DIR)
    print(f"SFT adapter saved -> {SFT_ADAPTER_DIR}")


## Post-SFT sanity check (3-sample greedy generation)

Before launching multi-hour GRPO, confirm the SFT adapter actually emits
`\boxed{}`. If this fails, GRPO will train on broken format and burn compute.


In [ ]:
if RUN_SFT:
    import torch
    model.eval()
    sample_rows = df_sft.sample(3, random_state=0)[["prompt", "answer"]].values.tolist()
    for p, ans in sample_rows:
        msgs = [{"role": "user", "content": str(p) + PROMPT_SUFFIX}]
        try:
            text = tokenizer.apply_chat_template(msgs, tokenize=False,
                                                  add_generation_prompt=True,
                                                  enable_thinking=True)
        except TypeError:
            text = tokenizer.apply_chat_template(msgs, tokenize=False,
                                                  add_generation_prompt=True)
        inputs = tokenizer(text, return_tensors="pt").to(model.device)
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=512, do_sample=False)
        gen = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:],
                               skip_special_tokens=False)
        has_box = "\\boxed{" in gen
        tag = "BOX  " if has_box else "NOBOX"
        print(f"[{tag}] expected={str(ans)!r:30s}  tail={gen[-180:]!r}")
    model.train()


## GRPO data prep (raw competition `train.csv`)

For GRPO we want **prompts + ground-truth answers only** — no pre-computed
CoT. The model generates rollouts; the binary verifier scores them.


In [ ]:
if RUN_GRPO:
    import pandas as pd
    from datasets import Dataset as HFDataset

    df_grpo = pd.read_csv(GRPO_DATA_PATH)
    print(f"GRPO data: {len(df_grpo)} rows.  Columns: {list(df_grpo.columns)}")
    if not {"prompt", "answer"}.issubset(df_grpo.columns):
        raise ValueError(f"train.csv must have 'prompt' and 'answer' columns; got {list(df_grpo.columns)}")

    df_grpo = df_grpo.dropna(subset=["prompt", "answer"]).reset_index(drop=True)
    df_grpo = df_grpo.sample(frac=1, random_state=SEED).reset_index(drop=True)

    records_g = []
    for _, row in df_grpo.iterrows():
        msgs = [{"role": "user", "content": str(row["prompt"]) + PROMPT_SUFFIX}]
        try:
            prompt_text = tokenizer.apply_chat_template(msgs, tokenize=False,
                                                        add_generation_prompt=True,
                                                        enable_thinking=True)
        except TypeError:
            prompt_text = tokenizer.apply_chat_template(msgs, tokenize=False,
                                                        add_generation_prompt=True)
        records_g.append({"prompt": prompt_text, "answer": str(row["answer"])})

    grpo_dataset = HFDataset.from_list(records_g)
    print(f"GRPO records: {len(records_g)}")


## Reward functions (binary RLVR)

Three components, summed:

1. **`format_reward`** — `+0.5` for any `\boxed{}`, `+0.3` for
   `<think>...</think>` *before* the boxed answer (blocks naked-boxed reward
   hacking).
2. **`accuracy_reward`** — `+2.0` if extracted answer matches ground truth
   (string-normalize OR relative numerical tolerance 1e-2).
3. **`overlong_penalty`** — DAPO soft penalty between
   `max_completion + buffer` and `2x max_completion`; hard `-1.0` beyond.

Total range roughly `[-1.0, +2.8]`. Binary-ish reward keeps GRPO advantage
estimates stable.


In [ ]:
if RUN_GRPO:
    import re

    _BOXED_RE = re.compile(r"\\boxed\{([^}]*)\}")
    _THINK_RE = re.compile(r"<think>.*?</think>", re.DOTALL)
    OVERLONG_BUFFER = 256

    def _content(c):
        # GRPOTrainer w/ conversational dataset: c is list[dict]; otherwise str.
        if isinstance(c, list) and c and isinstance(c[0], dict):
            return c[-1].get("content", "")
        return str(c)

    def _extract_boxed(text):
        # Brace-balanced parser (handles \boxed{\frac{1}{2}})
        idx = text.find("\\boxed{")
        if idx == -1:
            m = _BOXED_RE.search(text)
            return m.group(1).strip() if m else None
        depth, start = 1, idx + 7
        for i in range(start, len(text)):
            if text[i] == "{":
                depth += 1
            elif text[i] == "}":
                depth -= 1
            if depth == 0:
                return text[start:i].strip()
        return text[start:].strip()

    def _normalize(s):
        return str(s).strip().lower().replace(" ", "")

    def format_reward(completions, **kwargs):
        rewards = []
        for c in completions:
            text = _content(c)
            has_boxed = bool(_BOXED_RE.search(text))
            has_think = bool(_THINK_RE.search(text))
            think_before_boxed = (
                has_think and has_boxed and
                text.find("</think>") < text.rfind("\\boxed{")
            )
            r = 0.0
            if has_boxed:           r += 0.5
            if think_before_boxed:  r += 0.3
            rewards.append(r)
        return rewards

    def accuracy_reward(completions, answer, **kwargs):
        rewards = []
        for c, expected in zip(completions, answer):
            text = _content(c)
            predicted = _extract_boxed(text)
            ok = False
            if predicted is not None:
                if _normalize(predicted) == _normalize(expected):
                    ok = True
                else:
                    try:
                        pf, ef = float(predicted), float(expected)
                        if abs(pf - ef) <= 1e-2 * max(1.0, abs(ef)):
                            ok = True
                    except (ValueError, TypeError):
                        pass
            rewards.append(2.0 if ok else 0.0)
        return rewards

    def overlong_penalty(completions, **kwargs):
        rewards = []
        soft = GRPO_MAX_COMPLETION + OVERLONG_BUFFER
        hard = GRPO_MAX_COMPLETION * 2
        for c in completions:
            text = _content(c)
            n = len(text) / 4.0  # rough char->token estimate
            if   n <= soft: r = 0.0
            elif n >= hard: r = -1.0
            else:           r = -((n - soft) / (hard - soft))
            rewards.append(r)
        return rewards

    def combined_reward(completions, answer, **kwargs):
        f = format_reward(completions)
        a = accuracy_reward(completions, answer=answer)
        o = overlong_penalty(completions)
        return [x + y + z for x, y, z in zip(f, a, o)]

    # Tiny smoke test
    test = ["<think>\nblah\n</think>\n\\boxed{42}", "no boxed here"]
    test_ans = ["42", "42"]
    print("Smoke test (combined_reward):", combined_reward(test, answer=test_ans))
    print("Reward functions ready.")


## GRPO trainer (Dr. GRPO + DAPO knobs where supported)

Key choices (all from `CLAUDE.md` reference settings):

- `learning_rate=5e-6` — warm-started model, conservative
- `num_generations=8` — Dr. GRPO recipe minimum
- `temperature=1.0` — exploration at train time (eval still greedy)
- `beta=0.0` — no reference model, no KL → saves ~50% memory
- `scale_rewards=False` — Dr. GRPO: drop std normalization
- `loss_type="dapo"` — token-count normalization, no length bias
- `mask_truncated_completions=True` — exclude truncated from loss
- `epsilon_high=0.28` — DAPO clip-higher, prevents entropy collapse

TRL feature detection: capability flags applied only when the installed TRL
version supports them. With `trl>=0.16` all four kick in.


In [ ]:
if RUN_GRPO:
    import inspect, gc, time, torch
    from trl import GRPOTrainer, GRPOConfig

    _cfg_params = set(inspect.signature(GRPOConfig.__init__).parameters.keys())
    HAS_SCALE_REWARDS = "scale_rewards" in _cfg_params
    HAS_LOSS_TYPE     = "loss_type"     in _cfg_params
    HAS_MASK_TRUNC    = "mask_truncated_completions" in _cfg_params
    HAS_EPSILON_HIGH  = "epsilon_high"  in _cfg_params

    print(f"TRL capability flags:  scale_rewards={HAS_SCALE_REWARDS} "
          f"loss_type={HAS_LOSS_TYPE} mask_trunc={HAS_MASK_TRUNC} eps_high={HAS_EPSILON_HIGH}")

    grpo_kwargs = dict(
        output_dir                   = os.path.join(OUTPUT_DIR, "grpo_run"),
        num_train_epochs             = 1,
        per_device_train_batch_size  = 1,
        gradient_accumulation_steps  = 8,        # effective batch = 8 (divisible by num_generations)
        learning_rate                = 5e-6,
        lr_scheduler_type            = "cosine",
        warmup_steps                 = 10,
        max_prompt_length            = GRPO_MAX_PROMPT,
        max_completion_length        = GRPO_MAX_COMPLETION,
        num_generations              = 8,
        temperature                  = 1.0,
        top_p                        = 1.0,
        beta                         = 0.0,
        optim                        = "paged_adamw_8bit",
        adam_beta1                   = 0.9,
        adam_beta2                   = 0.999,
        weight_decay                 = 0.01,
        max_grad_norm                = 1.0,
        logging_steps                = 1,
        logging_dir                  = TB_LOG_DIR,
        report_to                    = "tensorboard",
        save_strategy                = "no",
        bf16                         = True,
        gradient_checkpointing       = True,
        gradient_checkpointing_kwargs= {"use_reentrant": False},
        seed                         = SEED,
        remove_unused_columns        = False,
    )

    # Apply Dr. GRPO / DAPO knobs only where TRL supports them
    if HAS_SCALE_REWARDS:
        grpo_kwargs["scale_rewards"] = False                # Dr. GRPO
    if HAS_LOSS_TYPE:
        grpo_kwargs["loss_type"] = "dapo"                   # DAPO token-count norm
    if HAS_MASK_TRUNC:
        grpo_kwargs["mask_truncated_completions"] = True
    if HAS_EPSILON_HIGH:
        grpo_kwargs["epsilon"]      = 0.2
        grpo_kwargs["epsilon_high"] = 0.28                  # DAPO clip-higher

    grpo_config = GRPOConfig(**grpo_kwargs)

    print("\n" + "=" * 60)
    print("  GRPO RLVR CONFIG (RTX 6000 Pro v13)")
    print("=" * 60)
    print(f"  LR:           {grpo_config.learning_rate}")
    print(f"  Num gen:      {grpo_config.num_generations}")
    print(f"  Beta (KL):    {grpo_config.beta}")
    print(f"  Max comp len: {grpo_config.max_completion_length}")
    print(f"  Temperature:  {grpo_config.temperature}")
    bs = grpo_config.per_device_train_batch_size
    ga = grpo_config.gradient_accumulation_steps
    print(f"  Batch:        {bs} x {ga} = {bs*ga} effective")
    print(f"  scale_rewards: {getattr(grpo_config, 'scale_rewards', 'n/a')}")
    print(f"  loss_type:     {getattr(grpo_config, 'loss_type', 'n/a')}")
    print(f"  mask_trunc:    {getattr(grpo_config, 'mask_truncated_completions', 'n/a')}")
    print("=" * 60 + "\n")

    grpo_trainer = GRPOTrainer(
        model            = model,
        args             = grpo_config,
        train_dataset    = grpo_dataset,
        reward_funcs     = [combined_reward],
        processing_class = tokenizer,
    )

    torch.cuda.empty_cache(); gc.collect()

    print("Starting GRPO RLVR training...")
    t0 = time.time()
    grpo_trainer.train()
    print(f"GRPO done in {(time.time()-t0)/60:.1f} min")

    os.makedirs(GRPO_ADAPTER_DIR, exist_ok=True)
    model.save_pretrained(GRPO_ADAPTER_DIR)
    tokenizer.save_pretrained(GRPO_ADAPTER_DIR)
    print(f"GRPO adapter saved -> {GRPO_ADAPTER_DIR}")


## Package `submission.zip`

Copies `adapter_config.json` + `adapter_model.safetensors` from the GRPO
adapter dir (or SFT if GRPO was skipped), patches `inference_mode=True`,
`lora_dropout=0.0`, and base model name. Zips with deflate.


In [ ]:
if RUN_PACKAGE:
    import json, shutil, zipfile

    src = GRPO_ADAPTER_DIR if RUN_GRPO else SFT_ADAPTER_DIR
    print("Packaging from:", src)
    os.makedirs(SUBMISSION_DIR, exist_ok=True)

    required = ["adapter_config.json", "adapter_model.safetensors"]
    for fname in required:
        sp = os.path.join(src, fname)
        dp = os.path.join(SUBMISSION_DIR, fname)
        if not os.path.exists(sp):
            raise FileNotFoundError(f"Missing: {sp}")
        shutil.copy2(sp, dp)
        print(f"  copied {fname}  ({os.path.getsize(dp)/1024/1024:.1f} MB)")

    cfg_path = os.path.join(SUBMISSION_DIR, "adapter_config.json")
    with open(cfg_path) as f:
        cfg = json.load(f)
    cfg["base_model_name_or_path"] = BASE_MODEL_NAME
    cfg["inference_mode"] = True
    cfg["lora_dropout"]   = 0.0
    with open(cfg_path, "w") as f:
        json.dump(cfg, f, indent=2)

    zip_path = os.path.join(OUTPUT_DIR, "submission.zip")
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        for fname in required:
            zf.write(os.path.join(SUBMISSION_DIR, fname), fname)
    print(f"\nsubmission.zip: {os.path.getsize(zip_path)/1024/1024:.1f} MB — ready.")


## Notes

- **First-run only**: install deps + restart kernel, then re-run cells.
- **Memory tuning**: if OOM during GRPO, drop `num_generations` 8 → 4, or
  `max_completion_length` 2048 → 1024.
- **Eval mismatch**: training runs at `temperature=1.0`; eval is `temperature=0.0`
  (greedy). This is expected — GRPO needs exploration; eval needs confidence.
- **Sanity check first**: after SFT, the 3-sample generation cell verifies
  `\boxed{}` is emitted. If not, fix the SFT data/formatter before running GRPO.
- **vLLM submission test**: run the submission zip through a local vLLM smoke
  test before uploading to Kaggle — `vLLM + LoRA + Mamba` is known fragile.
